In [1]:
# ================================
# Load Large Twitch Dataset (Custom Paths)
# ================================

import pandas as pd

# File paths (Windows paths with escaped backslashes)
edges_path = r"C:\Users\tuq24449\OneDrive - Temple University\Temple\2025\Spring '25\CIS 5524\Final Project\twitch_gamers\large_twitch_edges.csv"
features_path = r"C:\Users\tuq24449\OneDrive - Temple University\Temple\2025\Spring '25\CIS 5524\Final Project\twitch_gamers\large_twitch_features.csv"

# --- Load edges ---
try:
    edges = pd.read_csv(edges_path)
    print("✅ Edges loaded successfully.")
    print("📊 First 5 rows of edges (friendship links):")
    print(edges.head())
    print(f"🔗 Total number of edges: {len(edges)}\n")
except FileNotFoundError:
    print("❌ Could not find the edges file.")

# --- Load features ---
try:
    features_df = pd.read_csv(features_path, index_col=0)
    print("✅ Features loaded successfully.")
    print("📊 First 5 rows of features:")
    print(features_df.head())
    print(f"👤 Total number of users with features: {len(features_df)}\n")
except FileNotFoundError:
    print("❌ Could not find the features file.")

# --- Basic Info ---
print("🔍 Dataset Overview:")
print("Edge columns:", edges.columns.tolist())
print("Feature columns:", features_df.columns.tolist())


✅ Edges loaded successfully.
📊 First 5 rows of edges (friendship links):
   numeric_id_1  numeric_id_2
0         98343        141493
1         98343         58736
2         98343        140703
3         98343        151401
4         98343        157118
🔗 Total number of edges: 6797557

✅ Features loaded successfully.
📊 First 5 rows of features:
        mature  life_time  created_at  updated_at  numeric_id  dead_account  \
views                                                                         
7879         1        969  2016-02-16  2018-10-12           0             0   
500          0       2699  2011-05-19  2018-10-08           1             0   
382502       1       3149  2010-02-27  2018-10-12           2             0   
386          0       1344  2015-01-26  2018-10-01           3             0   
2486         0       1784  2013-11-22  2018-10-11           4             0   

       language  affiliate  
views                       
7879         EN          1  
500         

In [2]:
import networkx as nx

# Build an undirected graph from the edges
G = nx.from_pandas_edgelist(edges, source='numeric_id_1', target='numeric_id_2')

print(f"📈 Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")


📈 Graph created with 168114 nodes and 6797557 edges.


In [3]:
import random

# Sample positive edges from the graph
positive_pairs = random.sample(list(G.edges()), 10000)

# Create a fast lookup for all real edges
edge_set = set(map(tuple, map(sorted, G.edges())))

# Sample negative pairs (non-existent edges)
negative_pairs = set()
nodes = list(G.nodes())

while len(negative_pairs) < 10000:
    u, v = random.sample(nodes, 2)
    if (min(u, v), max(u, v)) not in edge_set:
        negative_pairs.add((u, v))

negative_pairs = list(negative_pairs)

print(f"✅ Sampled {len(positive_pairs)} positive and {len(negative_pairs)} negative pairs.")


✅ Sampled 10000 positive and 10000 negative pairs.


In [4]:
def extract_pair_features(u, v, features_df):
    u_data = features_df[features_df['numeric_id'] == u].squeeze()
    v_data = features_df[features_df['numeric_id'] == v].squeeze()

    year_u = pd.to_datetime(u_data['created_at']).year
    year_v = pd.to_datetime(v_data['created_at']).year

    return {
        'same_language': int(u_data['language'] == v_data['language']),
        'same_affiliate': int(u_data['affiliate'] == v_data['affiliate']),
        'same_mature': int(u_data['mature'] == v_data['mature']),
        'life_time_diff': abs(u_data['life_time'] - v_data['life_time']),
        'avg_life_time': (u_data['life_time'] + v_data['life_time']) / 2,
        'creation_year_diff': abs(year_u - year_v),
        'is_same_year_created': int(year_u == year_v),
        'language_combo': f"{u_data['language']}-{v_data['language']}"
    }


In [5]:
import pandas as pd

feature_rows = []

# Positive samples
for u, v in positive_pairs:
    row = extract_pair_features(u, v, features_df)
    row['label'] = 1
    feature_rows.append(row)

# Negative samples
for u, v in negative_pairs:
    row = extract_pair_features(u, v, features_df)
    row['label'] = 0
    feature_rows.append(row)

# Convert to DataFrame
pair_df = pd.DataFrame(feature_rows)
print("✅ Extracted features from all pairs.")
print(pair_df.head())


✅ Extracted features from all pairs.
   same_language  same_affiliate  same_mature  life_time_diff  avg_life_time  \
0              1               1            1             993         2032.5   
1              1               1            1             462         1054.0   
2              1               0            1             521         1917.5   
3              1               0            1             630         1984.0   
4              1               0            1             484         1791.0   

   creation_year_diff  is_same_year_created language_combo  label  
0                   3                     0          EN-EN      1  
1                   1                     0          EN-EN      1  
2                   2                     0          EN-EN      1  
3                   2                     0          EN-EN      1  
4                   1                     0          EN-EN      1  


In [6]:
def extract_graph_features(u, v, G):
    if not G.has_node(u) or not G.has_node(v):
        return {
            'num_common_neighbors': 0,
            'preferential_attachment': 0,
            'jaccard_similarity': 0,
            'adamic_adar': 0
        }

    common_neighbors = len(list(nx.common_neighbors(G, u, v)))
    degree_product = G.degree(u) * G.degree(v)

    try:
        jaccard = list(nx.jaccard_coefficient(G, [(u, v)]))[0][2]
    except:
        jaccard = 0

    try:
        adamic_adar = list(nx.adamic_adar_index(G, [(u, v)]))[0][2]
    except:
        adamic_adar = 0

    return {
        'num_common_neighbors': common_neighbors,
        'preferential_attachment': degree_product,
        'jaccard_similarity': jaccard,
        'adamic_adar': adamic_adar
    }


In [7]:
full_feature_rows = []

for i, row in pair_df.iterrows():
    u, v = positive_pairs[i] if row['label'] == 1 else negative_pairs[i - len(positive_pairs)]
    feature_dict = row.to_dict()
    feature_dict.update(extract_graph_features(u, v, G))
    full_feature_rows.append(feature_dict)

full_pair_df = pd.DataFrame(full_feature_rows)
print("✅ Graph-based features added.")
print(full_pair_df.head())


✅ Graph-based features added.
   same_language  same_affiliate  same_mature  life_time_diff  avg_life_time  \
0              1               1            1             993         2032.5   
1              1               1            1             462         1054.0   
2              1               0            1             521         1917.5   
3              1               0            1             630         1984.0   
4              1               0            1             484         1791.0   

   creation_year_diff  is_same_year_created language_combo  label  \
0                   3                     0          EN-EN      1   
1                   1                     0          EN-EN      1   
2                   2                     0          EN-EN      1   
3                   2                     0          EN-EN      1   
4                   1                     0          EN-EN      1   

   num_common_neighbors  preferential_attachment  jaccard_similarity  \
0 

In [8]:
from sklearn.model_selection import train_test_split

X_node = pair_df.drop(columns=['language_combo', 'label'])
y_node = pair_df['label']

X_node_train, X_node_test, y_node_train, y_node_test = train_test_split(
    X_node, y_node, test_size=0.2, random_state=42
)


In [9]:
X_full = full_pair_df.drop(columns=['language_combo', 'label'])
y_full = full_pair_df['label']

X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)


In [10]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split


In [23]:
# Node-only
X_node = pair_df.drop(columns=['language_combo', 'label'])
y_node = pair_df['label']
Xn_train, Xn_test, yn_train, yn_test = train_test_split(X_node, y_node, test_size=0.2, random_state=42)

# Node + Graph
X_full = full_pair_df.drop(columns=['language_combo', 'label'])
y_full = full_pair_df['label']
Xf_train, Xf_test, yf_train, yf_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)


In [25]:
# Node-only features
xgb_node = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_node.fit(Xn_train, yn_train)

yn_pred = xgb_node.predict(Xn_test)
yn_proba = xgb_node.predict_proba(Xn_test)[:, 1]

print("📊 XGBoost - Node Attributes Only")
print(classification_report(yn_test, yn_pred))
print("AUC:", roc_auc_score(yn_test, yn_proba))


# Node + Graph features
xgb_full = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_full.fit(Xf_train, yf_train)

yf_pred = xgb_full.predict(Xf_test)
yf_proba = xgb_full.predict_proba(Xf_test)[:, 1]

print("\n📊 XGBoost - Node + Graph Attributes")
print(classification_report(yf_test, yf_pred))
print("AUC:", roc_auc_score(yf_test, yf_proba))


C:\Users\tuq24449\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [12:08:57] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\tuq24449\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [12:08:57] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


📊 XGBoost - Node Attributes Only
              precision    recall  f1-score   support

           0       0.68      0.60      0.64      1981
           1       0.65      0.73      0.69      2019

    accuracy                           0.67      4000
   macro avg       0.67      0.67      0.66      4000
weighted avg       0.67      0.67      0.66      4000

AUC: 0.7381937469856654

📊 XGBoost - Node + Graph Attributes
              precision    recall  f1-score   support

           0       0.90      0.93      0.92      1981
           1       0.93      0.90      0.92      2019

    accuracy                           0.92      4000
   macro avg       0.92      0.92      0.92      4000
weighted avg       0.92      0.92      0.92      4000

AUC: 0.9700766744198666


In [27]:
import numpy as np
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from node2vec import Node2Vec

def evaluate_node2vec_sample(G, sample_size=3000, pair_count=10000):
    # Step 1: Sample nodes and subgraph
    sampled_nodes = random.sample(list(G.nodes()), sample_size)
    subG = G.subgraph(sampled_nodes).copy()
    
    try:
        node2vec = Node2Vec(subG, dimensions=64, walk_length=5, num_walks=5, workers=1)
        model = node2vec.fit(window=5, min_count=1)
    except:
        return None  # Skip if walk fails

    # Step 2: Sample positive and negative pairs
    edges_sub = set(map(tuple, map(sorted, subG.edges())))
    all_nodes = list(subG.nodes())

    positives = random.sample(list(edges_sub), min(pair_count // 2, len(edges_sub)))
    negatives = set()

    while len(negatives) < len(positives):
        u, v = random.sample(all_nodes, 2)
        if (min(u, v), max(u, v)) not in edges_sub:
            negatives.add((u, v))

    # Step 3: Build dataset with Hadamard product
    X, y = [], []
    for u, v in positives:
        try:
            vec = model.wv[str(u)] * model.wv[str(v)]
            X.append(vec)
            y.append(1)
        except KeyError:
            continue

    for u, v in negatives:
        try:
            vec = model.wv[str(u)] * model.wv[str(v)]
            X.append(vec)
            y.append(0)
        except KeyError:
            continue

    if len(X) < 1000:
        return None

    # Step 4: Train and evaluate
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = RandomForestClassifier()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    acc = clf.score(X_test, y_test)
    auc = roc_auc_score(y_test, y_proba)
    f1 = classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']

    return acc, f1, auc


In [53]:
results = []

for i in range(50):  # Run 5 times (adjust for time)
    print(f"🔁 Running sample {i+1}/50...")
    res = evaluate_node2vec_sample(G, sample_size=3000, pair_count=10000)
    if res:
        results.append(res)

if results:
    avg_acc = np.mean([r[0] for r in results])
    avg_f1 = np.mean([r[1] for r in results])
    avg_auc = np.mean([r[2] for r in results])

    print(f"\n📊 Averaged over {len(results)} samples:")
    print(f"✅ Accuracy: {avg_acc:.4f}")
    print(f"✅ F1-score: {avg_f1:.4f}")
    print(f"✅ AUC: {avg_auc:.4f}")
else:
    print("❌ All Node2Vec samples failed.")


🔁 Running sample 1/50...


Computing transition probabilities:   0%|          | 0/3000 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|█████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 35.61it/s]


🔁 Running sample 2/50...


Computing transition probabilities:   0%|          | 0/3000 [00:00<?, ?it/s]

...

Generating walks (CPU: 1): 100%|█████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.10it/s]


🔁 Running sample 50/50...


Computing transition probabilities:   0%|          | 0/3000 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|█████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 42.72it/s]



📊 Averaged over 50 samples:
✅ Accuracy: 0.9465
✅ F1-score: 0.9464
✅ AUC: 0.9889


In [37]:
import random

seed_nodes = random.sample(list(G.nodes()), 100)  # You can change 100


In [39]:
aa_scores = []

# Get top-scoring pairs for each seed node
for u in seed_nodes:
    neighbors = list(nx.non_neighbors(G, u))  # non-friends
    samples = random.sample(neighbors, min(300, len(neighbors)))  # limit for speed
    pairs = [(u, v) for v in samples]

    aa_scores.extend([
        (u, v, score)
        for u, v, score in nx.adamic_adar_index(G, pairs)
    ])


In [41]:
# Sort by AA score descending
aa_scores_sorted = sorted(aa_scores, key=lambda x: -x[2])

# Extract top node pairs (or flatten to nodes)
top_pairs = aa_scores_sorted[:5000]  # adjust for your size
top_nodes = set()

for u, v, _ in top_pairs:
    top_nodes.add(u)
    top_nodes.add(v)

print(f"🔍 Collected {len(top_nodes)} high-AA nodes.")


🔍 Collected 4975 high-AA nodes.


In [43]:
G_aa = G.subgraph(top_nodes).copy()

node2vec = Node2Vec(G_aa, dimensions=64, walk_length=5, num_walks=5, workers=1)
model = node2vec.fit(window=5, min_count=1)


Computing transition probabilities:   0%|          | 0/4975 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|█████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  2.49it/s]


In [45]:
# Get all nodes and edges in AA-based subgraph
all_nodes = list(G_aa.nodes())
edges_aa = set(map(tuple, map(sorted, G_aa.edges())))

# Sample positive (existing edges)
positive_aa = random.sample(list(edges_aa), min(5000, len(edges_aa)))

# Sample negative (non-edges)
negative_aa = set()
while len(negative_aa) < len(positive_aa):
    u, v = random.sample(all_nodes, 2)
    if (min(u, v), max(u, v)) not in edges_aa:
        negative_aa.add((u, v))


In [47]:
X = []
y = []

# Positive pairs
for u, v in positive_aa:
    try:
        vec = model.wv[str(u)] * model.wv[str(v)]
        X.append(vec)
        y.append(1)
    except KeyError:
        continue

# Negative pairs
for u, v in negative_aa:
    try:
        vec = model.wv[str(u)] * model.wv[str(v)]
        X.append(vec)
        y.append(0)
    except KeyError:
        continue

X = np.array(X)
y = np.array(y)


In [49]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print("📊 XGBoost on Node2Vec (Adamic-Adar Subgraph)")
print(classification_report(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))


C:\Users\tuq24449\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [12:15:19] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


📊 XGBoost on Node2Vec (Adamic-Adar Subgraph)
              precision    recall  f1-score   support

           0       0.87      0.88      0.87       988
           1       0.88      0.87      0.87      1012

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.87      0.87      0.87      2000

AUC: 0.9429437839048822


In [73]:
import random
import networkx as nx

def sample_mixed_aa_nodes(G, aa_ratio=0.5, total_nodes=3000):
    seed_nodes = random.sample(list(G.nodes()), 100)
    aa_scores = []

    for u in seed_nodes:
        neighbors = list(nx.non_neighbors(G, u))
        sample_neighbors = random.sample(neighbors, min(200, len(neighbors)))
        pairs = [(u, v) for v in sample_neighbors]

        aa_scores.extend([
            (u, v, score) for u, v, score in nx.adamic_adar_index(G, pairs)
        ])

    # Top Adamic-Adar nodes
    aa_scores_sorted = sorted(aa_scores, key=lambda x: -x[2])
    top_pairs = aa_scores_sorted[:int(total_nodes * aa_ratio)]

    aa_nodes = set()
    for u, v, _ in top_pairs:
        aa_nodes.add(u)
        aa_nodes.add(v)

    # Add random nodes to balance
    remaining_needed = total_nodes - len(aa_nodes)
    all_nodes = set(G.nodes()) - aa_nodes
    rand_nodes = set(random.sample(all_nodes, min(remaining_needed, len(all_nodes))))

    sampled_nodes = list(aa_nodes.union(rand_nodes))[:total_nodes]
    return sampled_nodes


In [75]:
X = []
y = []

# Positive pairs
for u, v in positive_aa:
    try:
        vec = model.wv[str(u)] * model.wv[str(v)]
        X.append(vec)
        y.append(1)
    except KeyError:
        continue

# Negative pairs
for u, v in negative_aa:
    try:
        vec = model.wv[str(u)] * model.wv[str(v)]
        X.append(vec)
        y.append(0)
    except KeyError:
        continue

X = np.array(X)
y = np.array(y)


In [77]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print("📊 XGBoost on Node2Vec (Adamic-Adar Subgraph)")
print(classification_report(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))


C:\Users\tuq24449\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [12:26:42] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


📊 XGBoost on Node2Vec (Adamic-Adar Subgraph)
              precision    recall  f1-score   support

           0       0.87      0.88      0.87       988
           1       0.88      0.87      0.87      1012

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.87      0.87      0.87      2000

AUC: 0.9429437839048822


In [85]:
import random
import networkx as nx

def sample_mixed_aa_nodes(G, aa_ratio=0.5, total_nodes=10000):
    seed_nodes = random.sample(list(G.nodes()), 100)
    aa_scores = []

    for u in seed_nodes:
        neighbors = list(nx.non_neighbors(G, u))
        sample_neighbors = random.sample(neighbors, min(200, len(neighbors)))
        pairs = [(u, v) for v in sample_neighbors]

        aa_scores.extend([
            (u, v, score) for u, v, score in nx.adamic_adar_index(G, pairs)
        ])

    # Top Adamic-Adar nodes
    aa_scores_sorted = sorted(aa_scores, key=lambda x: -x[2])
    top_pairs = aa_scores_sorted[:int(total_nodes * aa_ratio)]

    aa_nodes = set()
    for u, v, _ in top_pairs:
        aa_nodes.add(u)
        aa_nodes.add(v)

    # Add random nodes to balance
    remaining_needed = total_nodes - len(aa_nodes)
    all_nodes = set(G.nodes()) - aa_nodes
    rand_nodes = set(random.sample(all_nodes, min(remaining_needed, len(all_nodes))))

    sampled_nodes = list(aa_nodes.union(rand_nodes))[:total_nodes]
    return sampled_nodes


In [87]:
X = []
y = []

# Positive pairs
for u, v in positive_aa:
    try:
        vec = model.wv[str(u)] * model.wv[str(v)]
        X.append(vec)
        y.append(1)
    except KeyError:
        continue

# Negative pairs
for u, v in negative_aa:
    try:
        vec = model.wv[str(u)] * model.wv[str(v)]
        X.append(vec)
        y.append(0)
    except KeyError:
        continue

X = np.array(X)
y = np.array(y)


In [89]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print("📊 XGBoost on Node2Vec (Adamic-Adar Subgraph)")
print(classification_report(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))


C:\Users\tuq24449\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [12:28:55] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


📊 XGBoost on Node2Vec (Adamic-Adar Subgraph)
              precision    recall  f1-score   support

           0       0.87      0.88      0.87       988
           1       0.88      0.87      0.87      1012

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.87      0.87      0.87      2000

AUC: 0.9429437839048822


In [105]:
import random
import networkx as nx

def sample_mixed_cn_nodes(G, cn_ratio=0.5, total_nodes=3000):
    seed_nodes = random.sample(list(G.nodes()), 100)
    cn_scores = []

    for u in seed_nodes:
        neighbors = list(nx.non_neighbors(G, u))
        sample_neighbors = random.sample(neighbors, min(200, len(neighbors)))
        pairs = [(u, v) for v in sample_neighbors]

        for u, v in pairs:
            score = len(list(nx.common_neighbors(G, u, v)))
            if score > 0:
                cn_scores.append((u, v, score))

    # Sort by score
    cn_scores_sorted = sorted(cn_scores, key=lambda x: -x[2])
    top_pairs = cn_scores_sorted[:int(total_nodes * cn_ratio)]

    cn_nodes = set()
    for u, v, _ in top_pairs:
        cn_nodes.add(u)
        cn_nodes.add(v)


    # Add random nodes to balance
    remaining_needed = total_nodes - len(cn_nodes)
    all_nodes = set(G.nodes()) - cn_nodes
    rand_nodes = set(random.sample(sorted(all_nodes), min(remaining_needed, len(all_nodes))))


    sampled_nodes = list(cn_nodes.union(rand_nodes))[:total_nodes]
    return sampled_nodes


In [107]:
def evaluate_node2vec_on_cn(G, total_nodes=3000, pair_count=10000):
    sampled_nodes = sample_mixed_cn_nodes(G, total_nodes=total_nodes)
    subG = G.subgraph(sampled_nodes).copy()

    try:
        from node2vec import Node2Vec
        node2vec = Node2Vec(subG, dimensions=64, walk_length=5, num_walks=5, workers=1)
        model = node2vec.fit(window=5, min_count=1)
    except:
        return None

    # Sample positive and negative pairs
    edges_sub = set(map(tuple, map(sorted, subG.edges())))
    all_nodes = list(subG.nodes())

    positives = random.sample(list(edges_sub), min(pair_count // 2, len(edges_sub)))
    negatives = set()

    while len(negatives) < len(positives):
        u, v = random.sample(all_nodes, 2)
        if (min(u, v), max(u, v)) not in edges_sub:
            negatives.add((u, v))

    # Build features via Hadamard product
    X, y = [], []
    for u, v in positives:
        try:
            vec = model.wv[str(u)] * model.wv[str(v)]
            X.append(vec)
            y.append(1)
        except KeyError:
            continue

    for u, v in negatives:
        try:
            vec = model.wv[str(u)] * model.wv[str(v)]
            X.append(vec)
            y.append(0)
        except KeyError:
            continue

    if len(X) < 1000:
        return None

    # Train and evaluate
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, roc_auc_score

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = RandomForestClassifier()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    acc = clf.score(X_test, y_test)
    auc = roc_auc_score(y_test, y_proba)
    f1 = classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']

    return acc, f1, auc


In [109]:
results = []

for i in range(10):
    print(f"🔁 CN Sample {i+1}/10...")
    res = evaluate_node2vec_on_cn(G, total_nodes=3000)
    if res:
        results.append(res)

if results:
    avg_acc = np.mean([r[0] for r in results])
    avg_f1 = np.mean([r[1] for r in results])
    avg_auc = np.mean([r[2] for r in results])

    print(f"\n📊 Averaged over {len(results)} CN samples (3000 nodes):")
    print(f"✅ Accuracy: {avg_acc:.4f}")
    print(f"✅ F1-score: {avg_f1:.4f}")
    print(f"✅ AUC: {avg_auc:.4f}")
else:
    print("❌ All CN Node2Vec samples failed.")


🔁 CN Sample 1/10...


Computing transition probabilities:   0%|          | 0/3000 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|█████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 11.91it/s]


🔁 CN Sample 2/10...


Computing transition probabilities:   0%|          | 0/3000 [00:00<?, ?it/s]

...

Generating walks (CPU: 1): 100%|█████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  9.73it/s]


🔁 CN Sample 10/10...


Computing transition probabilities:   0%|          | 0/3000 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|█████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.71it/s]



📊 Averaged over 10 CN samples (3000 nodes):
✅ Accuracy: 0.9082
✅ F1-score: 0.9081
✅ AUC: 0.9696
